In [5]:
import numpy as np
import json
from openai import OpenAI
import pypdf
import os
import math
from dotenv import load_dotenv
from pathlib import Path

In [6]:
model_name = "gpt-5-mini"
embedding_model_name = "text-embedding-3-small"

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key is not None)

model_name = "gpt-5-mini"
embedding_model = "text-embedding-3-small"
client = OpenAI(api_key=api_key)

True


In [4]:
test_chunks = json.load(open("guidewire_chunks.json", "r"))

In [ ]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot_product / (norm_a * norm_b)

def search_chunks_by_embedding(chunks, query, top_k=5):
    query_response = client.embeddings.create(
        model=embedding_model,
        input=query
    )
    query_embedding = query_response.data[0].embedding

    results = []
    for chunk in chunks:
        score = cosine_similarity(query_embedding, chunk["embedding"])
        results.append({
            "file_name": chunk["file_name"],
            "page_number": chunk["page_number"],
            "text": chunk["text"],
            "score": score
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]